# Étape 3 — Robustesse hyperparamètres

**Date** : 2026-06-05  
**Statut** : Résultats — en attente de validation avant sélection finale  
**Prérequis** : Étape 2 terminée (grille 2×3), critère F2 acté (ADR-CF03)  
**Critère calibration seuil WF** : F2 (beta=2, rappel pondéré 4×), figé

## Question unique
Le classement de la grille étape 2 change-t-il quand on tune les hyperparamètres par cellule ?

## Protocole
- **Cellules** : APC_Global, APC_Haut, Enrichi_Seg (tuné via Enrichi_Haut)
- **Grille** : max_depth ∈ {4,6,8} × min_child_weight ∈ {1,5,10} × (n_estimators, lr) ∈ {(300,0.1),(500,0.05),(800,0.05)} = **27 combinaisons**
- **Sélection** : max WF PR-AUC haut (moyenne Fold1 + Fold2), H22-H24 uniquement
- **Évaluation** : H25 lecture seule, seuil F2 recalibré sur train
- **Comparaison** : vs étape 2 hyperparams figés ADR-38 (même seuil F2)


In [ ]:
import warnings, time, json, itertools
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import average_precision_score, r2_score
import xgboost as xgb
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('../../').resolve()
assert (PROJECT_ROOT / 'data').exists()
FIG = PROJECT_ROOT / 'consolidation_finale' / 'figures'
OUT = PROJECT_ROOT / 'consolidation_finale' / 'outputs'

SEUIL_UM     = 94
EMBARGO      = 9
CRITERE_BETA = 2   # F2 figé ADR-CF03
N_BOOT       = 1000
SEED         = 42

# Hyperparamètres figés ADR-38 (baseline étape 2)
XGB_BASE = dict(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, tree_method='hist',
    random_state=SEED, objective='reg:squarederror',
    enable_categorical=True, verbosity=0,
)

# Grille step 3 — subsample et colsample_bytree figés à 0.8 (non tuné ici)
GRID = list(itertools.product(
    [4, 6, 8],           # max_depth
    [1, 5, 10],          # min_child_weight
    [(300, 0.1), (500, 0.05), (800, 0.05)],  # (n_estimators, learning_rate)
))
print(f"Grille : {len(GRID)} combinaisons")

FOLDS = [
    {'name': 'Fold1', 'train': ['H22'],        'val': ['H23']},
    {'name': 'Fold2', 'train': ['H22', 'H23'], 'val': ['H24']},
]

plt.rcParams.update({'figure.dpi': 300, 'savefig.dpi': 300, 'font.size': 10,
    'axes.titlesize': 11, 'axes.labelsize': 10, 'legend.fontsize': 9,
    'figure.facecolor': 'white'})
PALETTE = sns.color_palette('colorblind', 6)
print('Setup OK')


In [ ]:
df = pd.read_parquet(PROJECT_ROOT / 'data/processed/ml_dataset.parquet')
assert 'H25' in df['horaire_annee_horaire'].unique()
print(f"Dataset : {len(df):,} lignes × {df.shape[1]} colonnes")

# Feature sets (identiques à étape 2)
CAL_FEATS   = [c for c in df.columns if c.startswith('cal_')]
EVENT_APC   = ['event_is_vacances_vaud','event_is_ferie_vaud','event_is_vacances_ou_ferie_vaud']
TOPO_FEATS  = ['spatial_gare_depart_ordre','spatial_gare_arrivee_ordre',
               'spatial_segment','base_duree_troncon_minutes']
HISTO_FEATS = [c for c in df.columns if c.startswith('histo_')]
APC_FEATS   = CAL_FEATS + EVENT_APC + TOPO_FEATS + HISTO_FEATS
assert len(APC_FEATS) == 38

with open(PROJECT_ROOT / 'models/11_modeling/xgb_B_prime_final_features.json') as f:
    ENRICHI_FEATS = json.load(f)['features']
assert len(ENRICHI_FEATS) == 179

print(f"APC-only : {len(APC_FEATS)} features")
print(f"Enrichi  : {len(ENRICHI_FEATS)} features")

# Partitions fixes
train_full = df[df['horaire_annee_horaire'].isin(['H22','H23','H24'])]
emb_h25    = df[df['horaire_annee_horaire']=='H25']['base_date'].min() + pd.Timedelta(EMBARGO,'D')
h25        = df[(df['horaire_annee_horaire']=='H25') & (df['base_date'] >= emb_h25)]
y_h25      = h25['target_voyageurs_2eme_classe'].values.astype('float64')
seg_h25    = h25['spatial_segment'].astype(str).values
print(f"Train H22-H24 : {len(train_full):,} | H25 eval : {len(h25):,}")
print(f"Surcharges haut H25 : {(h25[h25['spatial_segment']=='haut']['target_voyageurs_2eme_classe']>=SEUIL_UM).sum()}")


In [ ]:
def prep_X(df_sub, feats):
    X = df_sub[[f for f in feats if f in df_sub.columns]].copy()
    for c in X.columns:
        dt = str(X[c].dtype)
        if dt in ('object','string'): X[c] = X[c].astype('category')
        elif dt.startswith('Int') or dt.startswith('UInt'): X[c] = X[c].astype('float64')
    return X


def calibrer_f2(y_true, y_pred, seg, n_points=500):
    """Seuil maximisant F2 (beta=2) sur segment haut."""
    mask_h = seg == 'haut'
    yt_h, yp_h = y_true[mask_h], y_pred[mask_h]
    lb_h = (yt_h >= SEUIL_UM).astype(int)
    if lb_h.sum() == 0:
        return float(np.percentile(yp_h, 95))
    thresholds = np.linspace(np.percentile(yp_h, 70), np.percentile(yp_h, 99.9), n_points)
    best_t, best_f = thresholds[0], -1.
    for t in thresholds:
        pb = (yp_h >= t).astype(int)
        tp = int(((pb==1)&(lb_h==1)).sum()); fp = int(((pb==1)&(lb_h==0)).sum())
        fn = int(((pb==0)&(lb_h==1)).sum())
        if tp+fp == 0 or tp+fn == 0: continue
        p = tp/(tp+fp); r = tp/(tp+fn)
        if p+r == 0: continue
        fb = 5*p*r/(4*p+r)  # F2 = (1+4)*p*r/(4*p+r)
        if fb > best_f: best_f = fb; best_t = t
    return float(best_t)


def eval_wf_haut(y_true, y_pred, seg):
    """PR-AUC haut uniquement (pour sélection hyperparams WF)."""
    mask_h = seg == 'haut'
    lb_h   = (y_true[mask_h] >= SEUIL_UM).astype(int)
    return average_precision_score(lb_h, y_pred[mask_h]) if lb_h.sum() > 0 else 0.


def evaluer_h25(y_true, y_pred, seg, seuil, label):
    """Métriques complètes H25 avec bootstrap CI (haut segment)."""
    mask_h = seg == 'haut'
    yt_h, yp_h = y_true[mask_h], y_pred[mask_h]
    lb_h = (yt_h >= SEUIL_UM).astype(int)
    pr_auc = average_precision_score(lb_h, yp_h) if lb_h.sum() > 0 else 0.
    pb     = (yp_h >= seuil).astype(int)
    recall = pb[lb_h==1].sum() / lb_h.sum() if lb_h.sum() > 0 else 0.
    prec   = pb[lb_h==1].sum() / max(pb.sum(),1) if pb.sum() > 0 else 0.
    r2g    = r2_score(y_true, y_pred)
    r2h    = r2_score(yt_h, yp_h)
    rng = np.random.default_rng(SEED)
    prs, rcs = [], []
    for _ in range(N_BOOT):
        idx = rng.integers(0, len(yt_h), len(yt_h))
        lb_b = lb_h[idx]; yp_b = yp_h[idx]
        if lb_b.sum() == 0: continue
        prs.append(average_precision_score(lb_b, yp_b))
        pb_b = (yp_b >= seuil).astype(int)
        rcs.append(pb_b[lb_b==1].sum()/lb_b.sum())
    ci_pr = np.percentile(prs,[2.5,97.5]) if prs else [np.nan,np.nan]
    ci_rc = np.percentile(rcs,[2.5,97.5]) if rcs else [np.nan,np.nan]
    return {'cellule': label, 'seuil_wf': seuil,
            'pr_auc': pr_auc, 'pr_auc_ci_lo': ci_pr[0], 'pr_auc_ci_hi': ci_pr[1],
            'recall': float(recall), 'recall_ci_lo': ci_rc[0], 'recall_ci_hi': ci_rc[1],
            'precision': float(prec), 'r2_global': r2g, 'r2_haut': r2h,
            'n_surch_haut': int(lb_h.sum())}


def xgb_params_from_grid(depth, mcw, n_lr_pair):
    n_est, lr = n_lr_pair
    p = dict(XGB_BASE)
    p.update(max_depth=depth, min_child_weight=mcw, n_estimators=n_est, learning_rate=lr)
    return p


## Partie A — Walk-Forward tuning sur H22-H24

In [ ]:
# ── WF tuning : 27 combinaisons × 3 cellules × 2 folds ───────────────────────
# Cellules tunées :
#   APC_Global       : feats=APC,     train=all H22-H24 subset
#   APC_Haut         : feats=APC,     train=haut H22-H24 subset
#   Enrichi_Haut_Seg : feats=Enrichi, train=haut H22-H24 subset (→ Enrichi_Seg via recombination)

TUNE_CELLS = {
    'APC_Global':       {'feats': APC_FEATS,     'segment': 'all'},
    'APC_Haut':         {'feats': APC_FEATS,     'segment': 'haut'},
    'Enrichi_Haut_Seg': {'feats': ENRICHI_FEATS, 'segment': 'haut'},
}

wf_rows = []
total_runs = len(TUNE_CELLS) * len(GRID) * len(FOLDS)
run_idx = 0
t_start = time.time()

for cell_name, cell_cfg in TUNE_CELLS.items():
    feats   = cell_cfg['feats']
    seg_flt = cell_cfg['segment']

    for (depth, mcw, n_lr) in GRID:
        params = xgb_params_from_grid(depth, mcw, n_lr)
        fold_pr = []

        for fold in FOLDS:
            run_idx += 1
            # Données train
            tr_years = fold['train']
            va_years = fold['val']
            tr_all = df[df['horaire_annee_horaire'].isin(tr_years)]
            emb    = (df[df['horaire_annee_horaire']==va_years[0]]['base_date'].min()
                      + pd.Timedelta(EMBARGO,'D'))
            va_all = df[(df['horaire_annee_horaire'].isin(va_years)) & (df['base_date']>=emb)]

            # Filtrage segment si nécessaire
            if seg_flt == 'haut':
                tr = tr_all[tr_all['spatial_segment']=='haut']
                va = va_all[va_all['spatial_segment']=='haut']
            else:
                tr = tr_all; va = va_all

            X_tr = prep_X(tr, feats)
            y_tr = tr['target_voyageurs_2eme_classe'].values.astype('float64')
            X_va = prep_X(va, feats)
            y_va = va['target_voyageurs_2eme_classe'].values.astype('float64')
            seg_va = va['spatial_segment'].astype(str).values

            model = xgb.XGBRegressor(**params)
            model.fit(X_tr, y_tr)
            ypred = model.predict(X_va)

            pr_auc = eval_wf_haut(y_va, ypred, seg_va)
            seuil  = calibrer_f2(y_va, ypred, seg_va)
            fold_pr.append(pr_auc)

            wf_rows.append({
                'cellule': cell_name, 'fold': fold['name'],
                'max_depth': depth, 'min_child_weight': mcw,
                'n_estimators': n_lr[0], 'learning_rate': n_lr[1],
                'wf_pr_auc_haut': pr_auc, 'seuil_f2': seuil,
            })

            if run_idx % 30 == 0 or run_idx == total_runs:
                elapsed = time.time() - t_start
                eta = elapsed / run_idx * (total_runs - run_idx)
                print(f"[{run_idx}/{total_runs}] {cell_name} d={depth} mcw={mcw} "
                      f"n={n_lr[0]} lr={n_lr[1]} {fold['name']}: "
                      f"PR-AUC={pr_auc:.3f}  ETA={eta/60:.1f}min")

wf_df = pd.DataFrame(wf_rows)
wf_df.to_csv(OUT / 'etape_03_wf_tuning_results.csv', index=False)
print(f"\nTuning terminé en {(time.time()-t_start)/60:.1f}min")
print(f"Résultats : {len(wf_df)} lignes sauvegardées")


## Partie B — Sélection des meilleurs hyperparamètres

In [ ]:
# Sélection : max WF PR-AUC haut moyen (Fold1 + Fold2) par cellule
mean_wf = (wf_df.groupby(['cellule','max_depth','min_child_weight','n_estimators','learning_rate'])
           ['wf_pr_auc_haut'].mean().reset_index()
           .rename(columns={'wf_pr_auc_haut': 'mean_wf_pr_auc'}))

best_params = {}
print("=== Meilleurs hyperparamètres par cellule ===")
for cell_name in TUNE_CELLS:
    sub = mean_wf[mean_wf['cellule']==cell_name].sort_values('mean_wf_pr_auc', ascending=False)
    best = sub.iloc[0]
    best_params[cell_name] = {
        'max_depth': int(best['max_depth']),
        'min_child_weight': int(best['min_child_weight']),
        'n_estimators': int(best['n_estimators']),
        'learning_rate': float(best['learning_rate']),
        'mean_wf_pr_auc': float(best['mean_wf_pr_auc']),
    }
    # Baseline figé ADR-38 : d=6, mcw=1, n=300, lr=0.1
    base_row = sub[(sub['max_depth']==6)&(sub['min_child_weight']==1)&
                   (sub['n_estimators']==300)&(sub['learning_rate']==0.1)]
    base_pr  = float(base_row['mean_wf_pr_auc'].values[0]) if len(base_row) else np.nan
    print(f"\n{cell_name}:")
    print(f"  Figé ADR-38 (d=6,mcw=1,n=300,lr=0.1) : WF PR-AUC = {base_pr:.4f}")
    print(f"  Optimal tunée                          : WF PR-AUC = {best['mean_wf_pr_auc']:.4f}")
    print(f"  → max_depth={best['max_depth']}, min_child_weight={best['min_child_weight']}, "
          f"n_estimators={best['n_estimators']}, learning_rate={best['learning_rate']}")
    print(f"  Δ WF PR-AUC = {best['mean_wf_pr_auc']-base_pr:+.4f}")

# Sauvegarder best_params
import json as _json
with open(OUT / 'etape_03_best_params.json', 'w') as f:
    _json.dump(best_params, f, indent=2)
print("\nBest params sauvegardés → etape_03_best_params.json")


## Partie C — Évaluation H25 : hyperparamètres tunés

In [ ]:
# ── Entraînement final sur H22-H24 avec best hyperparams ─────────────────────
# Bug fix : calibrer_seuil_final évalue toujours sur la validation COMPLÈTE
# pour avoir le segment haut, même quand training sur bas uniquement.
# Le seuil F2 est toujours calibré sur le haut (où se trouvent les surcharges).

def calibrer_seuil_final(feats, train_seg, params):
    """
    Calibre seuil F2 sur le segment haut de la validation complète.
    Training peut être filtré (haut/bas/all) — seuil toujours calibré sur haut.
    """
    fold_seuils = []
    for fold in FOLDS:
        tr_all = df[df['horaire_annee_horaire'].isin(fold['train'])]
        emb    = (df[df['horaire_annee_horaire']==fold['val'][0]]['base_date'].min()
                  + pd.Timedelta(EMBARGO,'D'))
        va_all = df[(df['horaire_annee_horaire'].isin(fold['val'])) & (df['base_date']>=emb)]
        # Training sur le sous-ensemble
        if train_seg == 'haut':
            tr = tr_all[tr_all['spatial_segment']=='haut']
        elif train_seg == 'bas':
            tr = tr_all[tr_all['spatial_segment']=='bas']
        else:
            tr = tr_all
        # Prédiction sur la validation COMPLÈTE (haut inclus) pour calibrer le seuil
        model = xgb.XGBRegressor(**params)
        model.fit(prep_X(tr, feats), tr['target_voyageurs_2eme_classe'].values.astype('float64'))
        ypred = model.predict(prep_X(va_all, feats))
        seg_va = va_all['spatial_segment'].astype(str).values
        y_va   = va_all['target_voyageurs_2eme_classe'].values.astype('float64')
        s = calibrer_f2(y_va, ypred, seg_va)
        if not np.isnan(s):
            fold_seuils.append(s)
    return float(np.mean(fold_seuils)) if fold_seuils else np.nan


results_tuned = []

for cell_name, cell_cfg in TUNE_CELLS.items():
    feats   = cell_cfg['feats']
    seg_flt = cell_cfg['segment']
    bp      = best_params[cell_name]
    params_t = dict(XGB_BASE)
    params_t.update(max_depth=bp['max_depth'], min_child_weight=bp['min_child_weight'],
                    n_estimators=bp['n_estimators'], learning_rate=bp['learning_rate'])

    t0 = time.time()
    seuil_t = calibrer_seuil_final(feats, seg_flt, params_t)

    # Entraîner modèle principal sur H22-H24 (sous-ensemble si nécessaire)
    tr_final = train_full[train_full['spatial_segment']==seg_flt] if seg_flt != 'all' else train_full
    model_t = xgb.XGBRegressor(**params_t)
    model_t.fit(prep_X(tr_final, feats), tr_final['target_voyageurs_2eme_classe'].values.astype('float64'))
    ypred_t = model_t.predict(prep_X(h25, feats))

    if cell_name == 'Enrichi_Haut_Seg':
        # Modèle bas avec mêmes hyperparams (équité)
        tr_bas = train_full[train_full['spatial_segment']=='bas']
        model_bas = xgb.XGBRegressor(**params_t)
        model_bas.fit(prep_X(tr_bas, feats), tr_bas['target_voyageurs_2eme_classe'].values.astype('float64'))
        ypred_bas = model_bas.predict(prep_X(h25, feats))
        ypred_t = np.where(seg_h25=='haut', ypred_t, ypred_bas)
        label_display = 'Enrichi_Seg_tunée'
    elif cell_name == 'APC_Haut':
        label_display = 'APC_Haut_tunée'
    else:
        label_display = 'APC_Global_tunée'

    row = evaluer_h25(y_h25, ypred_t, seg_h25, seuil_t, label_display)
    results_tuned.append(row)
    print(f"{label_display}: seuil={seuil_t:.2f} PR-AUC={row['pr_auc']:.3f} "
          f"[{row['pr_auc_ci_lo']:.3f}–{row['pr_auc_ci_hi']:.3f}] "
          f"Rappel={row['recall']:.3f} Prec={row['precision']:.3f} R²={row['r2_global']:.3f}  "
          f"[{time.time()-t0:.0f}s]")

tuned_df = pd.DataFrame(results_tuned)


## Partie D — Baseline étape 2 (hyperparams figés ADR-38, seuil F2)

In [ ]:
# Recalcule les métriques H25 avec hyperparams figés + seuil F2
# (même baseline que etape_02bis, calculé ici pour cohérence complète)
print("=== Baseline étape 2 (figés ADR-38 : d=6,mcw=1,n=300,lr=0.1, seuil F2) ===")
results_base = []

base_configs = [
    ('APC_Global_figée',  APC_FEATS,     'all'),
    ('APC_Haut_figée',    APC_FEATS,     'haut'),
    ('Enrichi_Seg_figée', ENRICHI_FEATS, 'haut'),  # + bas pour Seg
]

for label, feats, seg_flt in base_configs:
    t0 = time.time()
    seuil_b = calibrer_seuil_final(feats, train_filt=seg_flt, params=XGB_BASE)

    if seg_flt == 'haut':
        tr_f = train_full[train_full['spatial_segment']=='haut']
    else:
        tr_f = train_full

    model_b = xgb.XGBRegressor(**XGB_BASE)
    model_b.fit(prep_X(tr_f, feats), tr_f['target_voyageurs_2eme_classe'].values.astype('float64'))
    ypred_b = model_b.predict(prep_X(h25, feats))

    if 'Seg' in label:
        # Recombiner Enrichi_Seg
        tr_bas = train_full[train_full['spatial_segment']=='bas']
        model_bas2 = xgb.XGBRegressor(**XGB_BASE)
        model_bas2.fit(prep_X(tr_bas, feats), tr_bas['target_voyageurs_2eme_classe'].values.astype('float64'))
        ypred_bas2 = model_bas2.predict(prep_X(h25, feats))
        ypred_b = np.where(seg_h25=='haut', ypred_b, ypred_bas2)

    row = evaluer_h25(y_h25, ypred_b, seg_h25, seuil_b, label)
    results_base.append(row)
    print(f"{label}: seuil={seuil_b:.2f} PR-AUC={row['pr_auc']:.3f} "
          f"[{row['pr_auc_ci_lo']:.3f}–{row['pr_auc_ci_hi']:.3f}] "
          f"Rappel={row['recall']:.3f} R²={row['r2_global']:.3f}  [{time.time()-t0:.0f}s]")

base_df = pd.DataFrame(results_base)


## Partie E — Comparaison tunée vs figée

In [ ]:
# ── Table comparaison ────────────────────────────────────────────────────────
# Paire chaque baseline avec sa version tunée
PAIRS = [
    ('APC_Global_figée',  'APC_Global_tunée',  'APC_Global'),
    ('APC_Haut_figée',    'APC_Haut_tunée',    'APC_Haut'),
    ('Enrichi_Seg_figée', 'Enrichi_Seg_tunée', 'Enrichi_Seg'),
]

print("=" * 95)
print(f"{'Cellule':18s} {'Critère':8s} {'Seuil':6s} {'PR-AUC':7s} {'IC 95%':15s} {'Rappel':7s} {'Prec':6s} {'R²glob':7s}")
print("-" * 95)

comparison_rows = []
for base_lbl, tuned_lbl, name in PAIRS:
    for lbl, vers in [(base_lbl, 'figée'), (tuned_lbl, 'tunée')]:
        src = base_df if vers=='figée' else tuned_df
        r = src[src['cellule']==lbl]
        if len(r) == 0: continue
        r = r.iloc[0]
        print(f"{name:18s} {vers:8s} {r['seuil_wf']:6.2f} {r['pr_auc']:7.3f} "
              f"[{r['pr_auc_ci_lo']:.3f}–{r['pr_auc_ci_hi']:.3f}]   "
              f"{r['recall']:7.3f} {r['precision']:6.3f} {r['r2_global']:7.3f}")
        comparison_rows.append({
            'cellule': name, 'version': vers,
            'pr_auc': r['pr_auc'], 'pr_auc_ci_lo': r['pr_auc_ci_lo'], 'pr_auc_ci_hi': r['pr_auc_ci_hi'],
            'recall': r['recall'], 'precision': r['precision'],
            'r2_global': r['r2_global'], 'seuil_wf': r['seuil_wf'],
            'n_surch_haut': r['n_surch_haut'],
        })
    print()

comp_df = pd.DataFrame(comparison_rows)
comp_df.to_csv(OUT / 'etape_03_tuned_vs_fixed.csv', index=False)
wf_df.to_csv(OUT / 'etape_03_wf_tuning_results.csv', index=False)
print("CSVs sauvegardés")


In [ ]:
# ── Figure 1 : PR-AUC tunée vs figée ─────────────────────────────────────────
cells_order = ['APC_Global','APC_Haut','Enrichi_Seg']
x = np.arange(len(cells_order)); w = 0.32
colors_fig = [PALETTE[4], PALETTE[0]]   # figée=gris-bleu, tunée=bleu

fig, axes = plt.subplots(1, 3, figsize=(15.1, 5.1))

metrics_plot = [
    ('pr_auc',    'pr_auc_ci_lo', 'pr_auc_ci_hi', 'PR-AUC haut (détection surcharges)'),
    ('recall',    'recall_ci_lo', 'recall_ci_hi',  'Rappel@WF haut (opérationnel)'),
    ('r2_global', None,           None,             'R² global H25'),
]
for ax, (met, lo, hi, title) in zip(axes, metrics_plot):
    vals_f, vals_t, errs_f, errs_t = [], [], [], []
    for cell in cells_order:
        sub_f = comp_df[(comp_df['cellule']==cell)&(comp_df['version']=='figée')]
        sub_t = comp_df[(comp_df['cellule']==cell)&(comp_df['version']=='tunée')]
        vf = sub_f[met].values[0] if len(sub_f) else np.nan
        vt = sub_t[met].values[0] if len(sub_t) else np.nan
        vals_f.append(vf); vals_t.append(vt)
        if lo:
            errs_f.append([vf-sub_f[lo].values[0], sub_f[hi].values[0]-vf] if len(sub_f) else [0,0])
            errs_t.append([vt-sub_t[lo].values[0], sub_t[hi].values[0]-vt] if len(sub_t) else [0,0])

    ax.bar(x-w/2, vals_f, w, label='Figés ADR-38', color=colors_fig[0], alpha=0.85)
    ax.bar(x+w/2, vals_t, w, label='Tunés étape 3', color=colors_fig[1], alpha=0.85)
    if lo:
        e_f = np.array(errs_f).T; e_t = np.array(errs_t).T
        ax.errorbar(x-w/2, vals_f, yerr=e_f, fmt='none', color='k', capsize=4, linewidth=1.5)
        ax.errorbar(x+w/2, vals_t, yerr=e_t, fmt='none', color='k', capsize=4, linewidth=1.5)
    ax.set_xticks(x); ax.set_xticklabels(cells_order, fontsize=9)
    ax.set_title(title); ax.set_ylabel(met.replace('_',' '))

p1 = mpatches.Patch(color=colors_fig[0], label='Figés ADR-38')
p2 = mpatches.Patch(color=colors_fig[1], label='Tunés étape 3')
fig.legend(handles=[p1,p2], loc='lower center', ncol=2, bbox_to_anchor=(0.5,-0.02))
fig.suptitle('Étape 3 — Robustesse hyperparamètres : tunés vs figés (H25, F2)', fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0,0.04,1,1])
fig.savefig(FIG/'etape_03_tuned_vs_fixed.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure 1 sauvegardée")

# ── Figure 2 : Sensibilité WF PR-AUC aux hyperparamètres ─────────────────────
fig2, axes2 = plt.subplots(1,3, figsize=(15,4.5))
for ax2, cell_name in zip(axes2, ['APC_Global','APC_Haut','Enrichi_Haut_Seg']):
    sub = mean_wf[mean_wf['cellule']==cell_name].sort_values('mean_wf_pr_auc', ascending=False)
    top = sub.head(15)
    labels = [f"d={int(r.max_depth)} mcw={int(r.min_child_weight)} n={int(r.n_estimators)} lr={r.learning_rate}"
              for _, r in top.iterrows()]
    colors_bar = ['gold' if i==0 else PALETTE[0] for i in range(len(top))]
    ax2.barh(range(len(top)-1,-1,-1), top['mean_wf_pr_auc'].values, color=colors_bar, alpha=0.85)
    ax2.set_yticks(range(len(top)-1,-1,-1)); ax2.set_yticklabels(labels, fontsize=7)
    ax2.set_title(f'{cell_name.replace("_Haut_Seg","_Seg")}
Top 15 combinaisons WF')
    ax2.set_xlabel('WF PR-AUC haut (moy. Fold1+Fold2)')
    # Ligne baseline ADR-38
    base_val = mean_wf[(mean_wf['cellule']==cell_name)&(mean_wf['max_depth']==6)&
                       (mean_wf['min_child_weight']==1)&(mean_wf['n_estimators']==300)&
                       (mean_wf['learning_rate']==0.1)]['mean_wf_pr_auc']
    if len(base_val):
        ax2.axvline(base_val.values[0], color='red', linestyle='--', linewidth=1.2,
                    label=f'ADR-38 ({base_val.values[0]:.3f})')
        ax2.legend(fontsize=8)
fig2.suptitle('Sensibilité WF PR-AUC haut aux hyperparamètres (H22-H24)', fontsize=11, fontweight='bold')
plt.tight_layout()
fig2.savefig(FIG/'etape_03_hyperparams_sensitivity.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure 2 sauvegardée")


## Verdict — Stabilité du classement

In [ ]:
# ── Verdict automatique ───────────────────────────────────────────────────────
print("=" * 70)
print("VERDICT ÉTAPE 3 — ROBUSTESSE HYPERPARAMÈTRES")
print("=" * 70)

# 1. Classement APC_Global vs Enrichi_Seg sur PR-AUC haut (tunée)
apc_g_t  = comp_df[(comp_df['cellule']=='APC_Global')  & (comp_df['version']=='tunée')]['pr_auc'].values
enr_s_t  = comp_df[(comp_df['cellule']=='Enrichi_Seg') & (comp_df['version']=='tunée')]['pr_auc'].values
apc_g_f  = comp_df[(comp_df['cellule']=='APC_Global')  & (comp_df['version']=='figée')]['pr_auc'].values
enr_s_f  = comp_df[(comp_df['cellule']=='Enrichi_Seg') & (comp_df['version']=='figée')]['pr_auc'].values

if len(apc_g_t) and len(enr_s_t):
    stable = (apc_g_t[0] > enr_s_t[0]) == (apc_g_f[0] > enr_s_f[0]) if len(apc_g_f) and len(enr_s_f) else None
    print(f"\nClassement PR-AUC haut — Figé:")
    if len(apc_g_f): print(f"  APC_Global   : {apc_g_f[0]:.3f}")
    if len(enr_s_f): print(f"  Enrichi_Seg  : {enr_s_f[0]:.3f}")
    print(f"\nClassement PR-AUC haut — Tunée:")
    print(f"  APC_Global   : {apc_g_t[0]:.3f}")
    print(f"  Enrichi_Seg  : {enr_s_t[0]:.3f}")
    print(f"\n→ Classement APC_Global > Enrichi_Seg : {'STABLE' if stable else 'INVERSÉ' if stable is False else 'N/A'}")

print()
for cell in ['APC_Global','APC_Haut','Enrichi_Seg']:
    sub_f = comp_df[(comp_df['cellule']==cell)&(comp_df['version']=='figée')]
    sub_t = comp_df[(comp_df['cellule']==cell)&(comp_df['version']=='tunée')]
    if not len(sub_f) or not len(sub_t): continue
    delta_pr = sub_t['pr_auc'].values[0] - sub_f['pr_auc'].values[0]
    delta_rc = sub_t['recall'].values[0] - sub_f['recall'].values[0]
    print(f"{cell:18s}: ΔPR-AUC={delta_pr:+.3f}  ΔRappel={delta_rc:+.3f}")
